In [ ]:
from huggingface_hub import login
login(token="")  # Paste self's access token in here

Check if access token is actually active:

In [2]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6aab8629f4552c1f3e73bd1f', 'name': 'tdgbcfxdzr', 'fullname': 'Bao Tran', 'isPro': False, 'avatarUrl': '/avatars/6dfa5ced89e7d43ce0c793d6a070f4bd.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'MedGemma', 'role': 'fineGrained', 'createdAt': '2026-09-22T03:51:21.267Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6aab8629f4552c1f3e73bd1f', 'type': 'user', 'name': 'tdgbcfxdzr'}, 'permissions': ['repo.content.read']}]}}}}


Check how much VRAM is available:

In [3]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.total,memory.used,memory.free", "--format=csv"], capture_output=True, text=True).stdout)

memory.total [MiB], memory.used [MiB], memory.free [MiB]
15360 MiB, 0 MiB, 14913 MiB



Import MedGemma from HuggingFace's model hub.

**Quantization** to shrink memory footprint in order to fit GPU's VRAM:


Note: Run `pip install torchao accelerate` before running this cell

In [4]:
!pip install torchao transformers -U

In [7]:
from transformers import pipeline, TorchAoConfig
from torchao.quantization import Int4WeightOnlyConfig

model_id = "google/medgemma-4b-it"

quantization_config = TorchAoConfig(Int4WeightOnlyConfig(group_size=32))

pipe = pipeline(
    task="image-text-to-text",
    model=model_id,
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[transformers] Gemma3ForConditionalGeneration LOAD REPORT from: google/medgemma-4b-it
Key                                                                  | Status     |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[transformers] Gemma3ForConditionalGeneration LOAD REPORT from: google/medgemma-4b-it
Key                                                                  | Status     |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

ValueError: Could not load model google/medgemma-4b-it with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForImageTextToText'>, <class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration'>). See the original errors:

while loading with AutoModelForImageTextToText, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/pipelines/base.py", line 240, in load_model
    model = model_class.from_pretrained(model, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/auto_factory.py", line 402, in from_pretrained
    return model_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *model_args, config=config, **hub_kwargs, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/m

If running the above cell leads to memory errors, run this instead (`pip install bitsandbytes`)

In [5]:
pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 33.6 MB/s eta 0:00:00


In [6]:
from transformers import pipeline, BitsAndBytesConfig
import torch

model_id = "google/medgemma-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

pipe = pipeline(
    task="image-text-to-text",
    model=model_id,
    model_kwargs={
        "quantization_config": quantization_config,
        "low_cpu_mem_usage": True,
    },
    device_map="auto",
)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Test model by feeding an image to it

In [7]:
from PIL import Image

image_path = "/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/Demos/medical_datasets/_tmp_dengue/images/PMC1/PMC10/PMC10225563_fimmu-14-1129246-g001_A_1_2.webp"
image = Image.open(image_path)   # raises a clear error immediately if the path is wrong

messages = [
    {"role": "system", "content": [{"type": "text", "text": "You are a helpful medical assistant."}]},
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Symptoms: fever, rash for 3 days. What's the likely diagnosis?"},
    ]},
]

output = pipe(text=messages, max_new_tokens=200)
print(output[0]["generated_text"][-1]["content"])

FileNotFoundError: [Errno 2] No such file or directory: '/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/Demos/medical_datasets/_tmp_dengue/images/PMC1/PMC10/PMC10225563_fimmu-14-1129246-g001_A_1_2.webp'